# DORA ICT Incident Classification & Vendor Compliance Demo

**DORA** (Digital Operational Resilience Act) requires EU financial institutions to classify ICT incidents, report within 4 hours, and maintain vendor contractual controls — and **prove compliance on demand**.

This notebook demonstrates factpy as a cross-domain auditable reasoning framework:
- 8 DORA rules encoded as formal Datalog
- 3 ICT incidents + 2 third-party vendors
- Full pipeline: evidence tree → certainty → provenance → audit → static HTML

## 1. Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve().parent / "src"))

import json, tempfile, shutil
from pprint import pprint

from factpy_kernel.sdk import SDKStore, Entity, Identity, Field, Rule, Pred, vars as sdk_vars
from factpy_kernel.sdk.dsl.rule import RuleRef
from factpy_kernel.authoring import FileAuthoringRegistry
from factpy_kernel.service.runtime_v1 import (
    open_runtime_session, close_runtime_session, reset_runtime_sessions_for_tests,
    write_runtime_fact, evaluate_runtime_derivation,
    explain_runtime_summary, explain_runtime_narrative, explain_runtime_nl,
    export_runtime_package, accept_runtime_derivation,
)
from factpy_kernel.audit import AuditQuery, load_audit_package
from factpy_kernel.audit.static_ui import render_audit_static_site

print("Imports OK")

Imports OK


## 2. Schema

- **ICTIncident**: affected clients, financial impact, duration, reporting status
- **DORAThreshold**: regulatory classification thresholds
- **ICTVendor**: contractual compliance attributes

In [2]:
class ICTIncident(Entity):
    incident_id: str = Identity(primary_key=True)
    locale: str = Identity()
    description: str = Field(cardinality="single")
    affected_clients: str = Field(cardinality="single")
    financial_impact_eur: str = Field(cardinality="single")
    duration_hours: str = Field(cardinality="single")
    reporting_status: str = Field(cardinality="single")

class DORAThreshold(Entity):
    threshold_id: str = Identity(primary_key=True)
    locale: str = Identity()
    client_threshold: str = Field(cardinality="single")
    financial_threshold_eur: str = Field(cardinality="single")
    duration_threshold_hours: str = Field(cardinality="single")

class ICTVendor(Entity):
    vendor_id: str = Identity(primary_key=True)
    locale: str = Identity()
    name: str = Field(cardinality="single")
    has_audit_rights: str = Field(cardinality="single")
    has_exit_strategy: str = Field(cardinality="single")
    has_subcontracting_controls: str = Field(cardinality="single")

sdk = SDKStore([ICTIncident, DORAThreshold, ICTVendor])
print(f"Schema: {len(sdk.schema_ir['predicates'])} predicates")

Schema: 21 predicates


## 3. Rules (8 DORA rules)

**Incident**: 3 threshold breaches + major classification (OR) + reporting ok/late
**Vendor**: compliant (all clauses) + non-compliant (missing exit strategy)

In [3]:
with sdk_vars("inc","thr","vendor","clients","amount","hours",
             "client_thr","amount_thr","duration_thr",
             "status","audit_rights","exit_strategy","subcontracting") as (
    inc,thr,vendor,clients,amount,hours,
    client_thr,amount_thr,duration_thr,
    status,audit_rights,exit_strategy,subcontracting):

    client_breach = Rule(id="q.dora_client_breach", version="1.0.0", select=[inc,clients],
        where=[Pred("ict_incident:affected_clients",inc,clients),
               Pred("dora_threshold:client_threshold",thr,client_thr),
               clients >= client_thr], expose=True,
        condition_weights={"b0.a0": 0.9, "b0.a1": 0.3})

    financial_breach = Rule(id="q.dora_financial_breach", version="1.0.0", select=[inc,amount],
        where=[Pred("ict_incident:financial_impact_eur",inc,amount),
               Pred("dora_threshold:financial_threshold_eur",thr,amount_thr),
               amount >= amount_thr], expose=True,
        condition_weights={"b0.a0": 0.8, "b0.a1": 0.4})

    duration_breach = Rule(id="q.dora_duration_breach", version="1.0.0", select=[inc,hours],
        where=[Pred("ict_incident:duration_hours",inc,hours),
               Pred("dora_threshold:duration_threshold_hours",thr,duration_thr),
               hours >= duration_thr], expose=True,
        condition_weights={"b0.a0": 0.7, "b0.a1": 0.5})

    major_incident = Rule(id="q.dora_major_incident", version="1.0.0", select=[inc,status],
        where=[[RuleRef(client_breach)(inc,clients), status == "major"],
               [RuleRef(financial_breach)(inc,amount), status == "major"],
               [RuleRef(duration_breach)(inc,hours), status == "major"]], expose=True)

    reporting_compliant = Rule(id="q.dora_reporting_compliant", version="1.0.0", select=[inc,status],
        where=[Pred("ict_incident:reporting_status",inc,status), status == "reported_within_4h"],
        expose=True, condition_weights={"b0.a0": 1.0})

    reporting_noncompliant = Rule(id="q.dora_reporting_noncompliant", version="1.0.0", select=[inc,status],
        where=[Pred("ict_incident:reporting_status",inc,status), status == "late_report"],
        expose=True, condition_weights={"b0.a0": 1.0})

    vendor_compliant = Rule(id="q.dora_vendor_compliant", version="1.0.0", select=[vendor,status],
        where=[Pred("ict_vendor:has_audit_rights",vendor,audit_rights), audit_rights == "yes",
               Pred("ict_vendor:has_exit_strategy",vendor,exit_strategy), exit_strategy == "yes",
               Pred("ict_vendor:has_subcontracting_controls",vendor,subcontracting), subcontracting == "yes",
               status == "compliant"], expose=True,
        condition_weights={"b0.a0": 0.8, "b0.a2": 0.9, "b0.a4": 0.7})

    vendor_noncompliant = Rule(id="q.dora_vendor_noncompliant_no_exit", version="1.0.0", select=[vendor,status],
        where=[Pred("ict_vendor:has_exit_strategy",vendor,exit_strategy), exit_strategy == "no",
               status == "non_compliant_missing_exit_strategy"], expose=True,
        condition_weights={"b0.a0": 1.0})

all_rules = [client_breach, financial_breach, duration_breach, major_incident,
             reporting_compliant, reporting_noncompliant, vendor_compliant, vendor_noncompliant]
print(f"{len(all_rules)} rules defined")
for r in all_rules: print(f"  {r.id}")

8 rules defined
  q.dora_client_breach
  q.dora_financial_breach
  q.dora_duration_breach
  q.dora_major_incident
  q.dora_reporting_compliant
  q.dora_reporting_noncompliant
  q.dora_vendor_compliant
  q.dora_vendor_noncompliant_no_exit


## 4. Registry + Data

In [4]:
registry_dir = tempfile.mkdtemp(prefix="dora_demo_")
registry = FileAuthoringRegistry(Path(registry_dir))
registry.upsert_schema_ir(sdk.schema_ir)
for rule in all_rules:
    registry.register_rule_spec(sdk._compile_rule_input(rule))

with sdk.batch() as tx:
    for iid, desc, cl, fi, dur, rep in [
        ("INC-042","Payment gateway outage","15000","2300000","6","reported_within_4h"),
        ("INC-043","Core banking degradation","45000","5100000","12","late_report"),
        ("INC-044","Minor email delay","200","5000","1","reported_within_4h")]:
        e = tx.entity(ICTIncident, incident_id=iid, locale="en")
        e.description.set(desc); e.affected_clients.set(cl)
        e.financial_impact_eur.set(fi); e.duration_hours.set(dur)
        e.reporting_status.set(rep)

    t = tx.entity(DORAThreshold, threshold_id="DORA-2025", locale="en")
    t.client_threshold.set("10000"); t.financial_threshold_eur.set("1000000")
    t.duration_threshold_hours.set("4")

    for vid, nm, ar, es, sc in [
        ("VENDOR-A","Acme Cloud Services","yes","yes","yes"),
        ("VENDOR-B","QuickPay Gateway","yes","no","yes")]:
        v = tx.entity(ICTVendor, vendor_id=vid, locale="en")
        v.name.set(nm); v.has_audit_rights.set(ar)
        v.has_exit_strategy.set(es); v.has_subcontracting_controls.set(sc)
    tx.commit()

print("Registry + data ready")

Registry + data ready


## 5. Runtime Session + Evaluate

In [ ]:
reset_runtime_sessions_for_tests()
sess = open_runtime_session({"registry_root": registry_dir})
session_id = sess["session"]["session_id"]

# Build entity display name mapping
entity_display = {}
incidents_meta = {}

# Rich metadata for every fact — source, analyst, method, confidence
INCIDENT_FACTS = {
    "INC-042": {
        "fields": [
            ("affected_clients", "15000", {"confidence": 0.92, "source": "Core Banking Transaction Log", "analyst": "M. Weber", "method": "automated_count", "assessed_at": "2026-03-15T14:30:00Z"}),
            ("financial_impact_eur", "2300000", {"confidence": 0.78, "source": "Finance Impact Assessment", "analyst": "K. Schmidt", "method": "manual_estimate", "assessed_at": "2026-03-16T09:00:00Z"}),
            ("duration_hours", "6", {"confidence": 0.95, "source": "Incident Timeline Report", "analyst": "Ops Team", "method": "system_log", "assessed_at": "2026-03-15T20:00:00Z"}),
            ("reporting_status", "reported_within_4h", {"confidence": 0.99, "source": "BaFin Notification System", "analyst": "Compliance Desk", "method": "automated_submission", "assessed_at": "2026-03-15T18:15:00Z"}),
        ],
    },
    "INC-043": {
        "fields": [
            ("affected_clients", "45000", {"confidence": 0.85, "source": "Customer Impact Analysis", "analyst": "L. Fischer", "method": "sampling_extrapolation", "assessed_at": "2026-03-17T11:00:00Z"}),
            ("financial_impact_eur", "5100000", {"confidence": 0.65, "source": "Preliminary Loss Estimate", "analyst": "K. Schmidt", "method": "manual_estimate", "assessed_at": "2026-03-18T10:00:00Z"}),
            ("duration_hours", "12", {"confidence": 0.90, "source": "Incident Timeline Report", "analyst": "Ops Team", "method": "system_log", "assessed_at": "2026-03-17T23:00:00Z"}),
            ("reporting_status", "late_report", {"confidence": 0.99, "source": "Compliance Audit Log", "analyst": "Compliance Desk", "method": "manual_review", "assessed_at": "2026-03-18T09:00:00Z"}),
        ],
    },
    "INC-044": {
        "fields": [
            ("affected_clients", "200", {"confidence": 0.98, "source": "Email Delivery Dashboard", "analyst": "IT Support", "method": "automated_count", "assessed_at": "2026-03-19T08:30:00Z"}),
            ("financial_impact_eur", "5000", {"confidence": 0.90, "source": "Standard Minor Incident Template", "analyst": "IT Support", "method": "template_estimate", "assessed_at": "2026-03-19T09:00:00Z"}),
            ("duration_hours", "1", {"confidence": 0.99, "source": "System Monitor", "analyst": "IT Support", "method": "system_log", "assessed_at": "2026-03-19T09:30:00Z"}),
            ("reporting_status", "reported_within_4h", {"confidence": 0.99, "source": "BaFin Notification System", "analyst": "Compliance Desk", "method": "automated_submission", "assessed_at": "2026-03-19T09:45:00Z"}),
        ],
    },
}

THRESHOLD_FACTS = [
    ("client_threshold", "10000", {"source": "DORA Art. 18(1)(a)", "version": "2025-01-17"}),
    ("financial_threshold_eur", "1000000", {"source": "DORA Art. 18(1)(b)", "version": "2025-01-17"}),
    ("duration_threshold_hours", "4", {"source": "DORA Art. 19(4)", "version": "2025-01-17"}),
]

VENDOR_FACTS = {
    "VENDOR-A": {
        "name": "Acme Cloud Services",
        "clauses": [
            ("has_audit_rights", "yes", {"source": "Contract ACM-2025-001 Section 8.3", "reviewed_by": "Legal Dept", "review_date": "2025-06-15"}),
            ("has_exit_strategy", "yes", {"source": "Contract ACM-2025-001 Section 12.1", "reviewed_by": "Legal Dept", "review_date": "2025-06-15"}),
            ("has_subcontracting_controls", "yes", {"source": "Contract ACM-2025-001 Section 9.2", "reviewed_by": "Legal Dept", "review_date": "2025-06-15"}),
        ],
    },
    "VENDOR-B": {
        "name": "QuickPay Gateway",
        "clauses": [
            ("has_audit_rights", "yes", {"source": "Contract QPG-2024-042 Section 7.1", "reviewed_by": "Legal Dept", "review_date": "2024-11-20"}),
            ("has_exit_strategy", "no", {"source": "Contract QPG-2024-042 — clause missing", "reviewed_by": "Legal Dept", "review_date": "2024-11-20", "note": "Exit strategy clause not present in current agreement"}),
            ("has_subcontracting_controls", "yes", {"source": "Contract QPG-2024-042 Section 8.4", "reviewed_by": "Legal Dept", "review_date": "2024-11-20"}),
        ],
    },
}

# Write incident facts with full metadata
for iid, data in INCIDENT_FACTS.items():
    ref = sdk.ref(ICTIncident, incident_id=iid, locale="en")
    entity_display[ref] = iid
    incidents_meta[ref] = {}
    for field_name, value, meta in data["fields"]:
        incidents_meta[ref][field_name] = value
        incidents_meta[ref][f"{field_name}_meta"] = meta
        write_runtime_fact(session_id, {
            "pred_id": f"ict_incident:{field_name}", "e_ref": ref,
            "rest_terms": [["string", value]],
            "meta": meta,
        }, kind="add")

# Write threshold facts with regulatory source metadata
thr_ref = sdk.ref(DORAThreshold, threshold_id="DORA-2025", locale="en")
entity_display[thr_ref] = "DORA-2025 Thresholds"
thresholds = {}
for field_name, value, meta in THRESHOLD_FACTS:
    thresholds[field_name] = value
    write_runtime_fact(session_id, {
        "pred_id": f"dora_threshold:{field_name}", "e_ref": thr_ref,
        "rest_terms": [["string", value]],
        "meta": meta,
    }, kind="add")

# Write vendor facts with contract reference metadata
vendors_meta = {}
for vid, vdata in VENDOR_FACTS.items():
    ref = sdk.ref(ICTVendor, vendor_id=vid, locale="en")
    entity_display[ref] = f"{vid}: {vdata['name']}"
    vendors_meta[ref] = {}
    for field_name, value, meta in vdata["clauses"]:
        vendors_meta[ref][field_name] = value
        vendors_meta[ref][f"{field_name}_meta"] = meta
        write_runtime_fact(session_id, {
            "pred_id": f"ict_vendor:{field_name}", "e_ref": ref,
            "rest_terms": [["string", value]],
            "meta": meta,
        }, kind="add")

def display_name(raw):
    if raw in entity_display: return entity_display[raw]
    for k, v in entity_display.items():
        if raw.endswith(k.split(":")[-1]): return v
    return raw[:20] + "..."

# Evaluate all derivations
results = {}
for drv_id, rule_id, target, v1, v2, label in [
    ("drv.major","q.dora_major_incident","ict_incident:reporting_status","$inc","$status","Major Incident"),
    ("drv.report_ok","q.dora_reporting_compliant","ict_incident:reporting_status","$inc","$status","Reporting OK"),
    ("drv.report_late","q.dora_reporting_noncompliant","ict_incident:reporting_status","$inc","$status","Reporting Late"),
    ("drv.vendor_ok","q.dora_vendor_compliant","ict_vendor:has_audit_rights","$vendor","$status","Vendor Compliant"),
    ("drv.vendor_bad","q.dora_vendor_noncompliant_no_exit","ict_vendor:has_exit_strategy","$vendor","$status","Vendor Non-Compliant")]:
    ev = evaluate_runtime_derivation(session_id, {"derivation":{
        "derivation_id":drv_id,"version":"1.0.0","target":target,
        "head_vars":[v1,v2],"where":[["ruleref",rule_id,"1.0.0",[v1,v2]]],"mode":"native"}})
    results[label] = ev
    n = len(ev["evaluation"]["candidates"]) if ev["ok"] else 0
    print(f"{label}: {n} candidates")

print(f"\nAll evaluations complete")
print(f"Every fact carries metadata: source, analyst/reviewer, method, confidence, date")

## 6. Visual Dashboard (data-driven from evaluation results)

In [ ]:
from IPython.display import HTML, display
from factpy_kernel.service.runtime_v1 import explain_runtime_tree

# ── Collect REAL data from evaluations ──
major_refs = set()
if results.get("Major Incident", {}).get("ok"):
    for c in results["Major Incident"]["evaluation"]["candidates"]:
        major_refs.add(c["payload"]["terms"][0]["value"])

reported_ok_refs = set()
if results.get("Reporting OK", {}).get("ok"):
    for c in results["Reporting OK"]["evaluation"]["candidates"]:
        reported_ok_refs.add(c["payload"]["terms"][0]["value"])

reported_late_refs = set()
if results.get("Reporting Late", {}).get("ok"):
    for c in results["Reporting Late"]["evaluation"]["candidates"]:
        reported_late_refs.add(c["payload"]["terms"][0]["value"])

vendor_ok_refs = set()
if results.get("Vendor Compliant", {}).get("ok"):
    for c in results["Vendor Compliant"]["evaluation"]["candidates"]:
        vendor_ok_refs.add(c["payload"]["terms"][0]["value"])

vendor_bad_refs = set()
if results.get("Vendor Non-Compliant", {}).get("ok"):
    for c in results["Vendor Non-Compliant"]["evaluation"]["candidates"]:
        vendor_bad_refs.add(c["payload"]["terms"][0]["value"])

# ── Build dashboard HTML from REAL results ──
CSS = """<style>
.dd{font-family:system-ui,-apple-system,sans-serif;max-width:920px}
.dd h2{font-size:1.3rem;font-weight:700;margin:16px 0 6px;color:#1a1a2e}
.dd .sub{color:#666;font-size:.83rem;margin-bottom:14px}
.dd .sec{margin:16px 0;padding:14px 16px;background:#fafafa;border-radius:10px;border:1px solid #eee}
.dd table{width:100%;border-collapse:collapse;font-size:.83rem}
.dd th{background:#f5f5f5;padding:7px 10px;text-align:left;font-weight:600;border-bottom:2px solid #ddd}
.dd td{padding:7px 10px;border-bottom:1px solid #eee}
.dd .ok{color:#1b5e20;font-weight:700} .dd .bad{color:#b71c1c;font-weight:700}
.dd .badge{display:inline-block;padding:2px 8px;border-radius:10px;font-size:.73rem;font-weight:700}
.dd .badge-ok{background:#c8e6c9;color:#1b5e20} .dd .badge-bad{background:#ffcdd2;color:#b71c1c}
.dd .grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(250px,1fr));gap:10px;margin:10px 0}
.dd .card{border-radius:8px;padding:12px 14px;border:1px solid #e0e0e0}
.dd .card-ok{background:linear-gradient(135deg,#e8f5e9,#f1f8e9);border-left:4px solid #2e7d32}
.dd .card-bad{background:linear-gradient(135deg,#ffebee,#fce4ec);border-left:4px solid #c62828}
.dd .card-blue{background:linear-gradient(135deg,#e3f2fd,#e8eaf6);border-left:4px solid #1565c0}
.dd .bar{height:7px;background:#e0e0e0;border-radius:4px;margin:3px 0;overflow:hidden}
.dd .bar-ok{background:linear-gradient(90deg,#66bb6a,#2e7d32)}
.dd .bar-bad{background:linear-gradient(90deg,#ef9a9a,#c62828)}
.dd .bar-warn{background:linear-gradient(90deg,#ffd54f,#f57f17)}
</style>"""

h = CSS + '<div class="dd">'
h += '<h2>DORA Compliance Dashboard</h2>'
h += '<div class="sub">All data below derived from formal reasoning evaluation results</div>'

# ── Incidents (from REAL evaluation + meta) ──
h += '<div class="sec"><h3 style="margin:0 0 8px;font-size:1rem">Incident Classification</h3>'
h += '<table><tr><th>Incident</th><th>Clients</th><th>Financial</th><th>Duration</th><th>Major?</th><th>4h Report?</th></tr>'
for ref, meta in incidents_meta.items():
    iid = display_name(ref)
    is_major = ref in major_refs
    is_ok = ref in reported_ok_refs
    is_late = ref in reported_late_refs
    cl = f'{int(meta["affected_clients"]):,}'
    fi = f'EUR {int(meta["financial_impact_eur"]):,}'
    dur = f'{meta["duration_hours"]}h'
    maj = '<span class="badge badge-bad">MAJOR</span>' if is_major else '<span class="badge badge-ok">MINOR</span>'
    rep = '<span class="ok">Within 4h</span>' if is_ok else ('<span class="bad">Late</span>' if is_late else '-')
    h += f'<tr><td><strong>{iid}</strong></td><td>{cl}</td><td>{fi}</td><td>{dur}</td><td>{maj}</td><td>{rep}</td></tr>'
h += '</table>'

# Threshold bars
h += '<div style="margin-top:10px;font-size:.75rem;color:#888">Thresholds: Clients >= 10,000 / Financial >= EUR 1,000,000 / Duration >= 4h</div>'
h += '<div class="grid">'
thr_vals = {"affected_clients": 10000, "financial_impact_eur": 1000000, "duration_hours": 4}
for ref, meta in incidents_meta.items():
    iid = display_name(ref)
    is_major = ref in major_refs
    card_cls = "card-bad" if is_major else "card-ok"
    h += f'<div class="card {card_cls}"><strong>{iid}</strong>'
    for field, thr_val in thr_vals.items():
        actual = int(meta[field])
        pct = min(actual / thr_val * 100, 100)
        bar_cls = "bar-bad" if actual >= thr_val else "bar-ok"
        label = field.replace("_", " ").title()
        sym = ">=" if actual >= thr_val else "<"
        h += f'<div style="font-size:.75rem;margin-top:3px">{label}: {actual:,} {sym} {thr_val:,}</div>'
        h += f'<div class="bar"><div class="{bar_cls}" style="width:{pct}%;height:100%;border-radius:4px"></div></div>'
    h += '</div>'
h += '</div></div>'

# ── Vendors (from REAL evaluation + meta) ──
h += '<div class="sec"><h3 style="margin:0 0 8px;font-size:1rem">Third-Party Vendor Compliance</h3>'
h += '<table><tr><th>Vendor</th><th>Audit Rights</th><th>Exit Strategy</th><th>Subcontracting</th><th>Status</th></tr>'
for ref, clauses in vendors_meta.items():
    vid = display_name(ref)
    is_ok = ref in vendor_ok_refs
    is_bad = ref in vendor_bad_refs
    def chk(v): return '<span class="ok">Yes</span>' if v == "yes" else '<span class="bad">No</span>'
    st = '<span class="badge badge-ok">COMPLIANT</span>' if is_ok else ('<span class="badge badge-bad">NON-COMPLIANT</span>' if is_bad else '-')
    h += f'<tr><td><strong>{vid}</strong></td><td>{chk(clauses["has_audit_rights"])}</td><td>{chk(clauses["has_exit_strategy"])}</td><td>{chk(clauses["has_subcontracting_controls"])}</td><td>{st}</td></tr>'
h += '</table></div>'

# ── Certainty (from REAL certainty_summary) ──
cert_cards = []
for label, ev in results.items():
    if not ev.get("ok"): continue
    for c in ev["evaluation"]["candidates"]:
        if c.get("confidence_kind") != "certainty": continue
        sm = explain_runtime_summary(session_id, {"kind":"candidate","id":c["candidate_id"]})
        if sm.get("ok") and sm.get("certainty_summary"):
            cs = sm["certainty_summary"]
            ent = display_name(c["payload"]["terms"][0]["value"])
            cert_cards.append({"label": label, "entity": ent, "cs": cs})

if cert_cards:
    h += '<div class="sec"><h3 style="margin:0 0 8px;font-size:1rem">Certainty Scores</h3><div class="grid">'
    for cc in cert_cards:
        cs = cc["cs"]
        agg = cs["aggregate_certainty"]
        color = "#2e7d32" if agg >= 0.7 else ("#ad6800" if agg >= 0.4 else "#c62828")
        card_cls = "card-ok" if agg >= 0.7 else ("card-blue" if agg >= 0.4 else "card-bad")
        h += f'<div class="card {card_cls}"><strong>{cc["label"]}</strong> ({cc["entity"]})'
        h += f'<div style="font-size:1.3rem;font-weight:700;color:{color};margin:4px 0">{agg}</div>'
        h += f'<div class="bar"><div style="width:{agg*100}%;height:100%;background:{color};border-radius:4px"></div></div>'
        h += f'<div style="font-size:.7rem;color:#888">{cs.get("aggregation","?")} | {len(cs.get("conditions",[]))} conditions</div>'
        for cond in cs.get("conditions",[]):
            w = cond.get("weight"); imp = cond.get("impact")
            if w is not None and imp is not None:
                is_bn = (cs.get("aggregation") == "bottleneck" and abs(imp - agg) < 0.001)
                bn = ' <span style="background:#ffcdd2;color:#b71c1c;padding:0 4px;border-radius:4px;font-size:.6rem">BOTTLENECK</span>' if is_bn else ''
                h += f'<div style="font-size:.7rem">{cond.get("atom_key","?")}: w={w} impact={imp}{bn}</div>'
        h += '</div>'
    h += '</div></div>'

# ── Summary counts ──
n_major = len(major_refs); n_late = len(reported_late_refs); n_vbad = len(vendor_bad_refs)
n_total = sum(len(ev["evaluation"]["candidates"]) for ev in results.values() if ev.get("ok"))
h += f'<div class="sec" style="background:linear-gradient(135deg,#e8eaf6,#e3f2fd)">'
h += f'<strong>{n_major}</strong> major incidents | <strong>{n_late}</strong> reporting breach | <strong>{n_vbad}</strong> vendor non-compliant | <strong>{n_total}</strong> total candidates with full audit trail'
h += '</div></div>'

display(HTML(h))

## 7. Deep Dive: Evidence Tree + Certainty + Narrative + NL

For each key candidate, we show the complete factpy analysis pipeline — all from real API calls, no hardcoded data.

In [ ]:
from factpy_kernel.service.runtime_v1 import explain_runtime_tree

NODE_STYLES = {
    "candidate_result":("Derived Result","#375a7f","#eef2f7"),
    "support_section":("Supporting Evidence","#375a7f","#eef2f7"),
    "predicate_witness_group":("Fact Match","#2e7d32","#e8f5e9"),
    "assertion_fact":("Witness Fact","#2e7d32","#f1f8e9"),
    "non_fact_check":("Constraint","#ad6800","#fff8e1"),
    "rule_ref_section":("Rule References","#7c4d9d","#f3e5f5"),
    "rule_ref":("Rule Invocation","#7c4d9d","#f3e5f5"),
    "referenced_support":("Child Proof","#7c4d9d","#ede7f6"),
    "unresolved_support":("Unresolved","#c62828","#ffebee"),
    "degraded_support":("Degraded","#757575","#f5f5f5"),
}

def render_tree_node(node, depth=0):
    kind = node.get("node_kind", "?")
    lbl, bc, bg = NODE_STYLES.get(kind, (kind, "#888", "#fafafa"))
    props = []
    if kind == "candidate_result":
        props.append(f"Result: <b>{node.get('root_result_kind','?')}</b>")
    elif kind == "support_section":
        props.append(f"{len(node.get('children',[]))} evidence items")
    elif kind == "predicate_witness_group":
        props.append(f"Predicate: <b>{node.get('pred_id','?')}</b>")
        props.append(f"Matching facts: {node.get('assertion_count','?')}")
        cc = node.get("condition_confidence")
        if cc is not None:
            c = "#2e7d32" if cc >= 0.8 else ("#ad6800" if cc >= 0.5 else "#c62828")
            props.append(f"Confidence: <b style='color:{c}'>{cc}</b> <span style='font-size:.65rem;color:#aaa'>(max of children)</span>")
    elif kind == "assertion_fact":
        claims = node.get("claim_args", [])
        if claims:
            val = ", ".join(cl.get("val","?") for cl in claims)
            props.append(f"Value: <b style='font-size:.95rem'>{val}</b>")
        conf = node.get("confidence")
        if conf is not None:
            c = "#2e7d32" if conf >= 0.8 else ("#ad6800" if conf >= 0.5 else "#c62828")
            props.append(f"Confidence: <b style='color:{c}'>{conf}</b> <span style='font-size:.65rem;color:#aaa'>(from fact meta)</span>")
        pred = node.get("pred_id")
        if pred: props.append(f"Predicate: {pred}")
        asrt = node.get("assertion_id","")
        if asrt: props.append(f"Assertion: <code style='font-size:.6rem;color:#aaa'>{asrt[:16]}...</code>")
    elif kind == "non_fact_check":
        ck = node.get("check_kind","?")
        st = node.get("status","?")
        sym = "pass" if st == "satisfied" else "FAIL"
        names = {"ruleref":"Rule reference","eq":"Equality","ge":">=","gt":">","le":"<="}
        props.append(f"{names.get(ck,ck)}: {sym}")
    elif kind == "rule_ref":
        props.append(f"Rule: <b>{node.get('rule_ref_id','?')}</b> v{node.get('rule_ref_version','?')}")
    elif kind == "referenced_support":
        d = node.get("support_digest","")
        if d: props.append(f"Digest: <code style='font-size:.6rem;color:#aaa'>{d[:20]}...</code>")
    elif kind == "rule_ref_section":
        props.append(f"{len(node.get('children',[]))} rules")

    ph = "".join(f"<div style='font-size:.75rem;margin:1px 0'>{p}</div>" for p in props)
    o = (f"<div style='border-left:3px solid {bc};background:{bg};border-radius:5px;"
         f"margin:{'1' if depth else '4'}px 0;padding:0;overflow:hidden;margin-left:{depth*12}px'>"
         f"<div style='padding:4px 8px'><b style='font-size:.78rem'>{lbl}</b>{ph}</div>")
    children = node.get("children",[])
    if children:
        o += "<div style='padding:0 4px 2px'>"
        for ch in children: o += render_tree_node(ch, depth+1)
        o += "</div>"
    return o + "</div>"

def render_candidate(cand, label):
    cid = cand["candidate_id"]
    terms = cand["payload"]["terms"]
    entity_ref = terms[0]["value"]
    result_val = terms[1]["value"] if len(terms) > 1 else "?"
    ent_name = display_name(entity_ref)
    ck = cand.get("confidence_kind", "none")
    is_bad = "non_compliant" in result_val or "late" in result_val
    bg = "#fff5f5" if is_bad else "#f0faf0"
    bbg = "#ffcdd2" if is_bad else "#c8e6c9"
    bco = "#b71c1c" if is_bad else "#1b5e20"
    rv = result_val.upper().replace("_"," ")

    h = f"<div style='background:{bg};border:1px solid #ddd;border-radius:10px;padding:14px 18px;margin:14px 0 6px'>"
    h += f"<div style='font-size:.7rem;font-weight:700;letter-spacing:.1em;text-transform:uppercase;color:#888'>{label}</div>"
    h += f"<div style='font-size:1.1rem;font-weight:700;margin:4px 0'>{ent_name}</div>"
    h += f"<span style='display:inline-block;padding:2px 8px;border-radius:10px;font-size:.78rem;font-weight:700;background:{bbg};color:{bco}'>{rv}</span>"
    h += f"<span style='margin-left:8px;font-size:.75rem;color:#888'>certainty routing: {ck}</span></div>"

    # Evidence tree
    tree_resp = explain_runtime_tree(session_id, {"kind": "candidate", "id": cid})
    if tree_resp.get("ok") and tree_resp.get("tree"):
        h += "<h4 style='margin:8px 0 4px;color:#375a7f'>Evidence Tree</h4>"
        h += render_tree_node(tree_resp["tree"]["root"])

    # Certainty
    sm = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid})
    if sm.get("ok"):
        cs = sm.get("certainty_summary")
        if cs:
            agg = cs["aggregate_certainty"]
            strat = cs.get("aggregation","?")
            color = "#2e7d32" if agg >= 0.7 else ("#ad6800" if agg >= 0.4 else "#c62828")
            cbg = "#e8f5e9" if agg >= 0.7 else ("#fff8e1" if agg >= 0.4 else "#ffebee")
            h += f"<div style='background:{cbg};border:1px solid #ddd;border-radius:8px;padding:10px 14px;margin:6px 0'>"
            h += f"<b>Certainty: <span style='font-size:1.2rem;color:{color}'>{agg}</span></b> <span style='font-size:.75rem;color:#888'>({strat})</span>"
            h += f"<div style='height:7px;background:#e0e0e0;border-radius:4px;margin:4px 0;overflow:hidden'><div style='width:{agg*100}%;height:100%;background:{color};border-radius:4px'></div></div>"
            for cond in cs.get("conditions",[]):
                w = cond.get("weight"); imp = cond.get("impact")
                if w is not None and imp is not None:
                    ic = "#2e7d32" if imp >= 0.7 else ("#ad6800" if imp >= 0.4 else "#c62828")
                    is_bn = (strat == "bottleneck" and abs(imp - agg) < 0.001)
                    bn = ' <span style="background:#ffcdd2;color:#b71c1c;padding:0 4px;border-radius:4px;font-size:.6rem">BOTTLENECK</span>' if is_bn else ''
                    h += f"<div style='display:flex;align-items:center;gap:6px;font-size:.73rem;margin:2px 0'>"
                    h += f"<span style='min-width:45px;color:#888'>{cond.get('atom_key','?')}</span>"
                    h += f"<span style='min-width:40px'>w={w}</span>"
                    h += f"<div style='flex:1;height:4px;background:#e0e0e0;border-radius:2px;overflow:hidden'>"
                    h += f"<div style='width:{imp*100}%;height:100%;background:{ic};border-radius:2px'></div></div>"
                    h += f"<span style='min-width:35px;text-align:right;font-weight:600'>{imp}</span>{bn}</div>"
            h += "</div>"

    # Narrative certainty lines
    narr = explain_runtime_narrative(session_id, {"kind": "candidate", "id": cid})
    if narr.get("ok"):
        cl = narr["narrative"].get("certainty_lines",[])
        if cl:
            h += "<h4 style='margin:8px 0 4px;color:#375a7f'>Certainty Narrative</h4>"
            for line in cl:
                lbg = "#ffebee" if "[bottleneck]" in line else "#f5f5f5"
                h += f"<div style='background:{lbg};border-radius:4px;padding:3px 8px;margin:1px 0;font-size:.75rem'>{line}</div>"

    # NL explanation
    nl = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid})
    if nl.get("ok"):
        paras = nl["explain_nl"]["paragraphs"]
        h += "<h4 style='margin:8px 0 4px;color:#375a7f'>Natural Language Explanation</h4>"
        for i, p in enumerate(paras, 1):
            dp = p
            for ref, name in entity_display.items():
                dp = dp.replace(ref, f"<b>{name}</b>")
            dp = dp.replace(cid, "<em>[this candidate]</em>")
            pbg = "#f3e5f5" if i == 5 else "#fafafa"
            h += f"<div style='background:{pbg};border-radius:5px;padding:6px 10px;margin:2px 0;font-size:.78rem'><span style='color:#aaa'>P{i}</span> {dp}</div>"

    return h

# Render showcase candidates
showcase = ["Major Incident", "Reporting Late", "Vendor Compliant", "Vendor Non-Compliant"]
full_html = '<div class="dd"><h2>Deep Dive: Full Candidate Analysis</h2>'
full_html += '<div class="sub">Evidence tree, certainty scoring, narrative, and NL explanation — all from real API calls</div>'

for lbl in showcase:
    ev = results.get(lbl, {})
    if ev.get("ok") and ev["evaluation"]["candidates"]:
        full_html += render_candidate(ev["evaluation"]["candidates"][0], lbl)
        full_html += "<hr style='border:none;border-top:2px solid #eee;margin:16px 0'>"

full_html += "</div>"
display(HTML(full_html))

## 8. Audit: Accept + Export + Static Site

In [ ]:
for ev in results.values():
    if ev["ok"]:
        for cand in ev["evaluation"]["candidates"]:
            accept_runtime_derivation(session_id, {"candidate": cand})

audit_dir = tempfile.mkdtemp(prefix="dora_audit_")
export_runtime_package(session_id, {"out_dir": audit_dir, "package_kind": "audit"})
pkg = load_audit_package(audit_dir)
aq = AuditQuery(pkg)
print(f"Audit candidates: {len(aq.list_candidates())}")
print(f"Provenance trees: {len(pkg.provenance_trees)}")

for cand_row in aq.list_candidates()[:3]:
    cid = cand_row["candidate_id"]
    pt = aq.get_candidate_provenance_tree(cid)
    cs = aq.get_candidate_certainty_summary(cid)
    print(f"\n  {cid[:30]}...")
    if cs: print(f"    certainty: {cs['aggregate_certainty']} ({cs['aggregation']})")
    if pt: print(f"    provenance: root={pt['root']['relation'][:30]}")
    else:  print(f"    provenance: (not materialized)")

site_dir = tempfile.mkdtemp(prefix="dora_site_")
render_audit_static_site(audit_dir, site_dir)
pages = list(Path(site_dir).rglob("*.html"))
print(f"\nStatic site: {len(pages)} pages")
print(f"To browse: python3 -m http.server 8199 --directory {site_dir}")

## Summary

| Check | INC-042 (15K clients, €2.3M, 6h) | INC-043 (45K clients, €5.1M, 12h) | INC-044 (200 clients, €5K, 1h) |
|-------|---|---|---|
| Major? | ✅ YES | ✅ YES | ❌ NO |
| Reported on time? | ✅ | ❌ late | ✅ |

| Vendor | Audit Rights | Exit Strategy | Subcontracting | Compliant? |
|--------|---|---|---|---|
| Acme Cloud | ✅ | ✅ | ✅ | ✅ YES |
| QuickPay | ✅ | ❌ | ✅ | ❌ NO |

**Cross-domain validated**: same auditable reasoning framework handles ECSS (space) + DORA (finance).